# Synthetic Gaming Data Generator

This notebook generates synthetic gaming data for a data engineering and analytics pipeline.

The generated data consists of three related datasets:

- **Users** — player account and demographic information.
- **Sessions** — player gameplay activity and session metrics.
- **Economy** — in-game currency and monetization transactions.

The datasets are generated with reproducible random values and exported as CSV files to the `../data` directory.

#### 1. Setup

Import the required libraries and configure the random seed to ensure that the generated data is reproducible across runs.

In [1]:
import numpy as np 
import pandas as pd
import os 
from pathlib import Path

# Set random seed for reporducibility 
np.random.seed(42)

output_dir = Path("../data")
os.makedirs(output_dir, exist_ok = True)

#### 2. Configuration

Define the main parameters used to control the size and characteristics of the synthetic dataset, including the number of users and the date range.

In [2]:
# Parameters
n_users = 10000
start_date = pd.to_datetime("2025-07-01")
end_date = pd.to_datetime("2026-07-01")

print("Generating player dimension table...")

Generating player dimension table...


#### 3. Generate Users

Generate the users dimension table containing player account information such as signup date, country, device type, and subscription status.

Each user is assigned a unique `user_id`.

In [3]:
#1 Dimension Table: Users 
user_ids = [f"U_{i:05d}" for i in range(1, n_users + 1)]
signup_dates = pd.date_range(start=start_date, end=end_date, periods=n_users)
countries = np.random.choice(
    ['Saudi Arabia', 'Kuwait', 'UAE', 'Bahrain', 'Qatar', 'Other'],
    size = n_users,
    p = [0.70, 0.10, 0.08, 0.05, 0.05, 0.02],
)
devices = np.random.choice(
    ['iOS', 'Andriod', 'Web'], 
    size = n_users,
    p = [0.40, 0.50, 0.10]
)
is_subscriber = np.random.choice(
    [0, 1],
    size = n_users,
    p = [0.88, 0.12]
) # 0 = "Regular Subsecriber" & 1 = "VIP Subscriber"
df_users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": signup_dates,
    "country": countries,
    "device": devices,
    "is_subscriber": is_subscriber,
})

output_file = os.path.join(output_dir, "dim_users.csv")
df_users.to_csv(output_file, index=False)

print("Generating session telemetry...")

Generating session telemetry...


In [4]:
df_users.head()

,user_id,signup_date,country,device,is_subscriber
0,U_00001,2025-07-01 00:00:00.000000,Saudi Arabia,iOS,0
1,U_00002,2025-07-01 00:52:33.915391,Qatar,iOS,0
2,U_00003,2025-07-01 01:45:07.830783,Kuwait,iOS,0
3,U_00004,2025-07-01 02:37:41.746174,Saudi Arabia,Andriod,0
4,U_00005,2025-07-01 03:30:15.661566,Saudi Arabia,Andriod,0


#### 4. Generate Sessions

Generate gameplay session records for the users.

Each session is linked to a user through `user_id` and contains information such as the session timestamp, duration, and number of games played.

In [5]:
# 2. Fact Table: Sessions (Multiple sessions per user over time)
session_records = []
for idx, row in df_users.iterrows():
    u_id = row['user_id']
    s_date = row['signup_date']

    # Active days probability based on user lifetime 
    active_days = np.random.randint(1, 45)
    for _ in range(active_days):
        session_date = s_date + pd.Timedelta(
            days=int(np.random.exponential(scale=10))
        )
        if session_date <= end_date:
            session_records.append({
                "session_id": f"S_{np.random.randint(1000000, 9999999)}",
                "user_id": u_id,
                "session_timestamp": session_date,
                "duration_minutes": np.random.gamma(shape=2, scale=15),
                "games_played": np.random.poisson(lam=4) +1,
            })

df_sessions = pd.DataFrame(session_records)

output_fact = os.path.join(output_dir, "fact_sessions.csv")
df_sessions.to_csv(output_fact, index=False)

print("Generating in-game economy transactions...")

Generating in-game economy transactions...


In [6]:
df_sessions.head()

,session_id,user_id,session_timestamp,duration_minutes,games_played
0,S_1981196,U_00001,2025-07-02,28.583526,2
1,S_9084602,U_00001,2025-07-15,7.573491,3
2,S_3847945,U_00001,2025-07-03,7.632640,6
3,S_1166544,U_00001,2025-07-15,49.122796,4
4,S_1916049,U_00001,2025-07-01,27.401491,2


#### 5. Generate Economy & Monetization Data

Generate in-game economy and monetization events based on player activity.

The generated events include:

- Coin sources
- Coin sinks
- In-app purchases

Each transaction is associated with a user and the relevant gameplay or monetization feature.

In [7]:
# 3. Fact Table: Economy & Monetization (Coin Sinks/Sources & Purchases)
tx_types = ['coin_sink', 'coin_source', 'iap_purchase']
features = ['Baloot Room', 'Tournament Entry', 'Store Bundle', 'Daily Reward']

economy_records = []
# Sample subset of sessions to generate economic events
sample_sessions = df_sessions.sample(frac=0.4)

for _, s_row in sample_sessions.iterrows():
    t_type = np.random.choice(
        tx_types, p=[0.45, 0.40, 0.15]
    ) # Balanced economy simulation
    amount = 0

    if t_type == 'iap_purchase':
        amount = np.random.choice([4.99, 9.99, 19.99, 49.99], p=[0.6, 0.25, 0.1, 0.05])
    elif t_type == 'coin_sink':
        amount = np.random.randint(100, 5000) # Coins spent on rooms/gifts 
    else: 
        amount = np.random.randint(200, 10000) # Coins won/earned 

    economy_records.append({
        "transaction_id": f"T_{np.random.randint(1000000, 9999999)}",
        "user_id": s_row["user_id"],
        "timestamp": s_row["session_timestamp"],
        "transaction_type": t_type,
        "feature_name": np.random.choice(features),
        "amount": amount,
    })

df_economy = pd.DataFrame(economy_records)

output_economy = os.path.join(output_dir, "fact_economy.csv")
df_economy.to_csv(output_economy, index=False)

In [8]:
df_economy.head()

,transaction_id,user_id,timestamp,transaction_type,feature_name,amount
0,T_2904775,U_03032,2025-11-05 15:25:17.551755,iap_purchase,Store Bundle,9.99
1,T_7560454,U_03532,2025-11-10 21:27:55.247524,coin_sink,Store Bundle,1478.00
2,T_6653275,U_00771,2025-08-18 02:35:14.851485,coin_sink,Tournament Entry,1746.00
3,T_3063874,U_04896,2026-02-02 16:26:55.841584,coin_source,Store Bundle,4171.00
4,T_1002479,U_08033,2026-04-20 04:44:08.424842,coin_sink,Tournament Entry,2167.00


In [9]:
print("Data pipeline executed successfully")
print(f"Generated files: {output_file}, {output_fact}, {output_economy}")

Data pipeline executed successfully
Generated files: ..\data\dim_users.csv, ..\data\fact_sessions.csv, ..\data\fact_economy.csv
